<a href="https://colab.research.google.com/github/acamogh/Gradio_Quiz_App/blob/main/Google_Cloud_Generative_AI_Leader_Practice_Exam_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =====================================================================
# CELL 1: ENVIRONMENT CONFIGURATION & PARSING (UPDATED)
# =====================================================================
print("📦 Installing mobile packages... (This takes about 40 seconds)")
import sys
import subprocess
import os
import glob
import json
import random  # Imported for random chunk selection later

# Install clean standalone packages
subprocess.run([sys.executable, "-m", "pip", "install", "--ignore-installed", "-q", "-U", "groq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "langchain-community", "pypdf", "tiktoken", "gradio"], check=True)
print("✅ Libraries installed successfully.")

import gradio as gr
from groq import Groq
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader

print("\n📂 Scanning notebook root for PDF files...")
pdf_files = glob.glob("*.pdf")

if not pdf_files:
    raise FileNotFoundError("⚠️ No PDF files found! Tap the folder icon on the left of Colab, upload your 8 PDFs, and run this cell again.")

print(f"📄 Found {len(pdf_files)} study guide files. Extracting content text...")

# CHANGED: We now store text in a list of chunks instead of one giant string
KNOWLEDGE_CHUNKS = []

for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        pages = loader.load()
        for page in pages:
            text = page.page_content.strip()
            if text:
                # Append each page as its own context chunk
                KNOWLEDGE_CHUNKS.append({
                    "source": pdf,
                    "content": text
                })
    except Exception as e:
        print(f"   Skipping or error reading {pdf}: {e}")

print(f"✅ Setup Complete. Loaded {len(KNOWLEDGE_CHUNKS)} individual pages into memory matrix.")

📦 Installing mobile packages... (This takes about 40 seconds)
✅ Libraries installed successfully.

📂 Scanning notebook root for PDF files...
📄 Found 8 study guide files. Extracting content text...
✅ Setup Complete. Loaded 588 individual pages into memory matrix.


In [8]:
# =====================================================================
# CELL 2: STATE ENGINE, QUOTA TRACKER, & PARALLEL MOBILE LAYOUT
# =====================================================================
import random
import json
import os
from datetime import datetime

class MobileQuizEngine:
    def __init__(self, knowledge_chunks):
        self.chunks = knowledge_chunks
        self.current_quiz = None
        self.incorrect_log = []

        # High Volume Strategy Configuration
        self.model_id = 'openai/gpt-oss-120b'
        self.backup_model_id = 'openai/gpt-oss-20b'
        self.active_model_used = "None"

        # Daily Quota Tracking Architecture
        self.daily_request_allowance = 500
        self.requests_used_today = 0

        # Track expected answer count for the active question locally
        self.expected_answer_count = 1

        self.history_file = "genai_study_history.json"
        self.load_history_from_disk()

        try:
            api_key = userdata.get('GROQ_API_KEY')
            self.groq_client = Groq(api_key=api_key)
        except Exception:
            raise ValueError("❌ GROQ_API_KEY missing. Set it in Colab's Secrets menu (Key Icon).")

    def load_history_from_disk(self):
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, dict):
                        self.incorrect_log = data.get("mistakes", [])
                        self.requests_used_today = data.get("requests_used_today", 0)
                    else:
                        self.incorrect_log = data
                print(f"🔄 Loaded {len(self.incorrect_log)} total historical mistakes from disk storage.")
            except Exception as e:
                print(f"⚠️ Could not read history file, starting fresh: {e}")
                self.incorrect_log = []

    def save_history_to_disk(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump({
                    "mistakes": self.incorrect_log,
                    "requests_used_today": self.requests_used_today
                }, f, indent=4)
        except Exception as e:
            print(f"⚠️ Failed to save history to disk: {e}")

    def get_remaining_requests(self):
        return max(0, self.daily_request_allowance - self.requests_used_today)

    def generate_question(self):
        review_context = ""
        if self.incorrect_log and len(self.incorrect_log) % 3 == 0:
            past_mistake = random.choice(self.incorrect_log)
            review_context = f"""
            CRITICAL REVIEW CONTEXT: The student previously failed a question matching this target concept: '{past_mistake['concept']}'.
            Do NOT reuse this old question text: '{past_mistake['question_text']}'.
            Instead, heavily paraphrase the scenario, alter the variables entirely, and address the concept from a completely different angle.
            """

        selected_chunks = random.sample(self.chunks, min(3, len(self.chunks)))
        dynamic_context = ""
        for idx, chunk in enumerate(selected_chunks):
            dynamic_context += f"\n--- Context Source {idx+1}: File '{chunk['source']}', Page {chunk['page']} ---\n"
            dynamic_context += chunk['content'] + "\n"

        chosen_style = random.choices(["single", "multi"], weights=[70, 30], k=1)[0]

        if chosen_style == "single":
            self.expected_answer_count = 1
            style_instruction = """
            STYLE ENFORCEMENT: You MUST generate a standard SINGLE-SELECT question.
            - There must be exactly ONE logically correct option.
            - The 'correct_answer' field in your JSON output MUST be a single letter string (e.g., 'A').
            """
        else:
            self.expected_answer_count = random.choice([2, 3])
            style_instruction = f"""
            STYLE ENFORCEMENT: You MUST generate a MULTI-SELECT scenario question.
            - Explicitly end your question text with instructions on how many items to check (e.g., 'Select the TWO options that...' or 'Choose THREE components...').
            - There must be exactly {self.expected_answer_count} correct options.
            - The 'correct_answer' field in your JSON output MUST be a comma-separated list of values sorted alphabetically (e.g., 'A, C' or 'B, C, D').
            """

        system_instruction = f"""
        You are an expert exam simulator for the Google Cloud Generative AI Leader certification.

        Knowledge Base Material with Reference Sources:
        {dynamic_context}

        Instructions:
        1. NEVER copy exact sentences or verbatim questions from the source text.
        2. Generate one highly challenging multiple-choice question matching the official certification level.
        3. Provide exactly four options (A, B, C, D).
        4. Identify which specific source file and page number you primarily based the question on.

        {style_instruction}
        {review_context}

        Output Structure Requirement: Return data STRICTLY inside a standard JSON object matching this structural layout:
        {{
            "question": "Clear scenario text goes here.",
            "options": {{
                "A": "Option text content",
                "B": "Option text content",
                "C": "Option text content",
                "D": "Option text content"
            }},
            "correct_answer": "Format based on enforcement rule above",
            "concept": "The underlying technical principle tested",
            "explanation": "Detailed professional rationale breakdown describing why the choices are correct and others are wrong.",
            "source_file": "The exact name of the PDF file used",
            "source_page": "The exact page number used as an integer or string"
        }}
        """

        try:
            response = self.groq_client.chat.completions.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.8
            )
            self.active_model_used = "🤖 Premium Engine (gpt-oss-120b)"
            self.requests_used_today += 1
            self.save_history_to_disk()
        except Exception as e:
            print(f"⚡ Primary model rate-limit hit. Switching to high-volume gpt-oss-20b...")
            response = self.groq_client.chat.completions.create(
                model=self.backup_model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.7
            )
            self.active_model_used = "⚡ Fast Fallback Workhorse (gpt-oss-20b)"
            self.requests_used_today += 1
            self.save_history_to_disk()

        try:
            raw_content = response.choices[0].message.content
            self.current_quiz = json.loads(raw_content)
            return self.current_quiz
        except Exception as e:
            print(f"Error parsing JSON output: {e}")
            return None

    def log_failure(self):
        if self.current_quiz:
            file_ref = self.current_quiz.get("source_file", "Unknown")
            page_ref = self.current_quiz.get("source_page", "?")

            self.incorrect_log.append({
                "concept": self.current_quiz.get("concept"),
                "question_text": self.current_quiz.get("question"),
                "source": f"{file_ref} (Pg. {page_ref})",
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
            })
            self.save_history_to_disk()

    def teach_weakest_concept(self):
        if not self.incorrect_log:
            return "### 🎓 Tutor Note\nYou haven't made any mistakes yet! Go take some questions first so I know what to teach you."

        concept_counts = {}
        for item in self.incorrect_log:
            c = item["concept"]
            concept_counts[c] = concept_counts.get(c, 0) + 1
        weakest_concept = max(concept_counts, key=concept_counts.get)
        fail_count = concept_counts[weakest_concept]

        relevant_text_pool = ""
        matched_chunks = [c for c in self.chunks if weakest_concept.lower() in c["content"].lower()]

        if not matched_chunks:
            matched_chunks = random.sample(self.chunks, min(3, len(self.chunks)))

        for chunk in matched_chunks[:3]:
            relevant_text_pool += f"\n[Context from {chunk['source']} Page {chunk['page']}]:\n{chunk['content']}\n"

        tutor_prompt = f"""
        You are an elite corporate technical instructor for the Google Cloud Generative AI Leader Certification.
        The student is highly struggling with the core concept: '{weakest_concept}'. They have failed questions on this topic {fail_count} times.

        Using the background reference material from their textbook below, write an engaging, deeply insightful, clear text-book lesson that breaks down this topic completely.

        Reference Material:
        {relevant_text_pool}

        Lesson Layout Requirements:
        1. 💎 **The Concept Core**: Explain the concept simply but professionally.
        2. ⚙️ **How it works in Google Cloud**: Relate it directly to GCP architectural rules, tools, or best practices.
        3. ⚡ **Exam Trap Warning**: Highlight *exactly* how certification questions will try to trick them on this specific topic (the technical nuances).
        4. 📝 **Quick Memory Checklist**: Provide 3 bullet points that summarize everything they must remember.
        """

        try:
            response = self.groq_client.chat.completions.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": "You are a professional Google Cloud AI teaching assistant."},
                    {"role": "user", "content": tutor_prompt}
                ],
                temperature=0.7
            )
            self.requests_used_today += 1
            self.save_history_to_disk()
            return response.choices[0].message.content
        except Exception as e:
            response = self.groq_client.chat.completions.create(
                model=self.backup_model_id,
                messages=[{"role": "system", "content": tutor_prompt}],
                temperature=0.6
            )
            self.requests_used_today += 1
            self.save_history_to_disk()
            return response.choices[0].message.content

    def get_summary(self):
        if not self.incorrect_log:
            return "### 🎉 Perfect Record!\nNo weak areas have been logged yet."

        concept_counts = {}
        for item in self.incorrect_log:
            c = item["concept"]
            concept_counts[c] = concept_counts.get(c, 0) + 1

        summary = "## 📋 Complete Master List of Your Mistakes\n"
        summary += "Here is your historical record of missed concepts across all sessions:\n\n"

        for concept, count in sorted(concept_counts.items(), key=lambda x: x[1], reverse=True):
            status_icon = "🚨" if count > 2 else "⚠️"
            summary += f"* {status_icon} **{concept}** — Failed `{count}` times total.\n"

        return summary

# Instantiate active engine links
quiz_engine = MobileQuizEngine(KNOWLEDGE_CHUNKS)

# =====================================================================
# GRADIO INTERFACE FUNCTIONS
# =====================================================================
def load_new_question():
    data = quiz_engine.generate_question()
    if not data:
        return (
            "❌ API Timeout. Click 'Fetch Next Question' to retry.",
            gr.update(visible=False), gr.update(visible=False),
            "", "", gr.Button(interactive=False)
        )

    q_text = f"### Question:\n{data['question']}"
    file_name = data.get('source_file', 'Unknown PDF')
    page_num = data.get('source_page', 'Unknown Page')

    meta_text = f"📖 **Source:** `{file_name}` (Pg. `{page_num}`) | 🧠 **Model:** `{quiz_engine.active_model_used}` | ⏳ **Remaining Daily Balance:** `{quiz_engine.get_remaining_requests()}` calls"

    choices_list = [
        f"A: {data['options'].get('A', '')}",
        f"B: {data['options'].get('B', '')}",
        f"C: {data['options'].get('C', '')}",
        f"D: {data['options'].get('D', '')}"
    ]

    if quiz_engine.expected_answer_count == 1:
        radio_update = gr.update(choices=choices_list, value=None, label="👇 Select exactly 1 option:", visible=True)
        checkbox_update = gr.update(choices=[], value=[], label="", visible=False)
    else:
        label_suffix = "options"
        dynamic_label = f"👇 Select exactly {quiz_engine.expected_answer_count} {label_suffix}:"
        radio_update = gr.update(choices=[], value=None, label="", visible=False)
        checkbox_update = gr.update(choices=choices_list, value=[], label=dynamic_label, visible=True)

    return q_text, radio_update, checkbox_update, "", meta_text, gr.Button(interactive=False)

def check_button_readiness(selected_choices):
    if not selected_choices:
        return gr.Button(interactive=False, variant="secondary")
    actual_count = 1 if isinstance(selected_choices, str) else len(selected_choices)
    if actual_count == quiz_engine.expected_answer_count:
        return gr.Button(interactive=True, variant="primary")
    else:
        return gr.Button(interactive=False, variant="secondary")

def evaluate_selection(radio_choice, checkbox_choices):
    if not quiz_engine.current_quiz:
        return "⚠️ Please load a question first."

    active_selection = [radio_choice] if quiz_engine.expected_answer_count == 1 else checkbox_choices
    if not active_selection or (quiz_engine.expected_answer_count > 1 and not checkbox_choices) or (quiz_engine.expected_answer_count == 1 and not radio_choice):
        return "⚠️ Complete option parameter selection bounds before attempting validation."

    user_letters = sorted([c.split(":")[0].strip().upper() for c in active_selection if c])
    user_answer_string = ", ".join(user_letters)

    correct_raw = quiz_engine.current_quiz['correct_answer'].strip().upper()
    correct_letters = sorted([x.strip() for x in correct_raw.split(",")])
    correct_answer_string = ", ".join(correct_letters)

    if user_answer_string == correct_answer_string:
        score_text = "### ✅ CORRECT!\n\n"
    else:
        file_name = quiz_engine.current_quiz.get('source_file', 'Unknown PDF')
        page_num = quiz_engine.current_quiz.get('source_page', 'Unknown Page')
        score_text = f"### ❌ INCORRECT.\nThe correct answer selection was: **({correct_answer_string})**.\nYour response was: *({user_answer_string})*.\n\n📍 *Double check this on page {page_num} of {file_name}*\n\n"
        quiz_engine.log_failure()

    return f"{score_text}📝 **Rationale Solution Breakdown**:\n{quiz_engine.current_quiz['explanation']}"

def view_summary_panel():
    summary_data = quiz_engine.get_summary()
    quota_header = f"### 📊 Quota Overview Today\n* You have consumed `{quiz_engine.requests_used_today}` total generation operations.\n* You have `{quiz_engine.get_remaining_requests()}` remaining calls remaining in your safe daily study budget.\n\n---\n"
    return f"{quota_header}{summary_data}"

def trigger_tutor_lesson():
    return quiz_engine.teach_weakest_concept()

# =====================================================================
# GRADIO CUSTOM CSS SKINNING (GOOGLE SANS OVERLAY CODE)
# =====================================================================
google_font_css = """
@import url('https://fonts.googleapis.com/css2?family=Open+Sans:ital,wght@0,300..800;1,300..800&family=Product+Sans:ital,wght@0,100..900;1,100..900&display=swap');

.gradio-container, .gradio-container * {
    font-family: 'Product Sans', 'Open Sans', system-ui, -apple-system, sans-serif !important;
}
"""

# =====================================================================
# APPLICATION INTERFACE GENERATION CANVAS
# =====================================================================
with gr.Blocks(theme=gr.themes.Soft(), css=google_font_css) as mobile_app:
    gr.Markdown("# 🧠 Google Cloud GenAI Leader App Hub")

    with gr.Group():
        question_display = gr.Markdown("### Tap '➡️ Fetch Next Question' to begin your practice run!")

    choice_radio = gr.Radio(choices=[], label="👇 Select your choice:", visible=False)
    choice_checkboxes = gr.CheckboxGroup(choices=[], label="👇 Select your choices:", visible=False)

    # CHANGED: Action buttons are now organized side-by-side inside a parallel Row layout
    with gr.Row():
        submit_btn = gr.Button("🔒 Submit Selection", variant="secondary", interactive=False)
        next_btn = gr.Button("➡️ Fetch Next Question", variant="secondary")

    # CHANGED: Feedback Markdown container is pushed immediately below the primary action controls row
    feedback_display = gr.Markdown("")

    with gr.Row():
        summary_btn = gr.Button("📊 Weak Summary Dashboard", variant="stop")
        teach_btn = gr.Button("🏫 Teach Me My Weakest Concept", variant="primary")

    metadata_tracker = gr.Markdown("")

    choice_radio.change(fn=check_button_readiness, inputs=[choice_radio], outputs=[submit_btn])
    choice_checkboxes.change(fn=check_button_readiness, inputs=[choice_checkboxes], outputs=[submit_btn])

    next_btn.click(
        fn=load_new_question,
        inputs=[],
        outputs=[question_display, choice_radio, choice_checkboxes, feedback_display, metadata_tracker, submit_btn],
        queue=True
    )

    submit_btn.click(
        fn=evaluate_selection,
        inputs=[choice_radio, choice_checkboxes],
        outputs=[feedback_display]
    )

    summary_btn.click(fn=view_summary_panel, inputs=[], outputs=[feedback_display])
    teach_btn.click(fn=trigger_tutor_lesson, inputs=[], outputs=[feedback_display])

mobile_app.launch(inline=False, share=True, debug=False)

/tmp/ipykernel_4435/923730746.py:350: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=google_font_css) as mobile_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b090975704559b25ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
# =====================================================================
# CELL 2: STATE ENGINE & TARGETED GRADIO CANVAS 111
# =====================================================================
import random
import json

class MobileQuizEngine:
    def __init__(self, knowledge_chunks):
        self.chunks = knowledge_chunks
        self.incorrect_log = []
        self.current_quiz = None

        # High Volume Strategy Configuration
        self.model_id = 'openai/gpt-oss-120b'
        self.backup_model_id = 'openai/gpt-oss-20b'
        self.active_model_used = "None"

        try:
            api_key = userdata.get('GROQ_API_KEY')
            self.groq_client = Groq(api_key=api_key)
        except Exception:
            raise ValueError("❌ GROQ_API_KEY missing. Set it in Colab's Secrets menu (Key Icon).")

    def generate_question(self):
        review_context = ""
        if self.incorrect_log and len(self.incorrect_log) % 3 == 0:
            past_mistake = self.incorrect_log[-1]
            review_context = f"""
            CRITICAL REVIEW CONTEXT: The student previously failed a question matching this target concept: '{past_mistake['concept']}'.
            Do NOT reuse this old question text: '{past_mistake['question_text']}'.
            Instead, heavily paraphrase the scenario, alter the variables entirely, and address the concept from a completely different angle.
            """

        selected_chunks = random.sample(self.chunks, min(3, len(self.chunks)))

        dynamic_context = ""
        for idx, chunk in enumerate(selected_chunks):
            dynamic_context += f"\n--- Context Source {idx+1}: File '{chunk['source']}', Page {chunk['page']} ---\n"
            dynamic_context += chunk['content'] + "\n"

        system_instruction = f"""
        You are an expert exam simulator for the Google Cloud Generative AI Leader certification.

        Knowledge Base Material with Reference Sources:
        {dynamic_context}

        Instructions:
        1. NEVER copy exact sentences or verbatim questions from the source text.
        2. Generate one highly challenging multiple-choice question matching the official certification level.
        3. Provide exactly four options (A, B, C, D) with strictly ONE correct answer.
        4. Do NOT reveal clues or answers inside your fields.
        5. Identify which specific source file and page number you primarily based the question on.

        {review_context}

        Output Structure Requirement: Return data STRICTLY inside a standard JSON object matching this structural layout:
        {{
            "question": "Clear scenario/question text goes here...",
            "options": {{
                "A": "Option text content",
                "B": "Option text content",
                "C": "Option text content",
                "D": "Option text content"
            }},
            "correct_answer": "A, B, C, or D",
            "concept": "The underlying technical principle tested",
            "explanation": "Detailed professional rationale breakdown describing why the choice is correct and others are wrong.",
            "source_file": "The exact name of the PDF file used",
            "source_page": "The exact page number used as an integer or string"
        }}
        """

        try:
            response = self.groq_client.chat.completions.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.8
            )
            self.active_model_used = "🤖 Premium Engine (gpt-oss-120b)"

        except Exception as e:
            print(f"⚡ Primary model rate-limit/timeout hit. Switching to high-volume gpt-oss-20b... Details: {e}")
            response = self.groq_client.chat.completions.create(
                model=self.backup_model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question based on the provided context."}
                ],
                response_format={"type": "json_object"},
                temperature=0.7
            )
            self.active_model_used = "⚡ Fast Fallback Workhorse (gpt-oss-20b)"

        try:
            raw_content = response.choices[0].message.content
            self.current_quiz = json.loads(raw_content)
            return self.current_quiz
        except Exception as e:
            print(f"Error parsing JSON output: {e}")
            return None

    def log_failure(self):
        if self.current_quiz:
            file_ref = self.current_quiz.get("source_file", "Unknown")
            page_ref = self.current_quiz.get("source_page", "?")
            self.incorrect_log.append({
                "concept": self.current_quiz.get("concept"),
                "question_text": self.current_quiz.get("question"),
                "source": f"{file_ref} (Pg. {page_ref})"
            })

    def get_summary(self):
        if not self.incorrect_log:
            return "🎉 Perfect record! No weak areas logged yet."
        summary = "### 📊 Weak Concepts to Review:\n"
        for idx, item in enumerate(self.incorrect_log, 1):
            summary += f"{idx}. **Domain Focus**: {item['concept']} *(Source: {item['source']})*\n"
        return summary

# Instantiate active engine links
quiz_engine = MobileQuizEngine(KNOWLEDGE_CHUNKS)

# =====================================================================
# GRADIO INTERFACE FUNCTIONS
# =====================================================================
def load_new_question():
    data = quiz_engine.generate_question()
    if not data:
        # Returns an empty string fallback for the new metadata tracker panel
        return "❌ API Timeout. Click 'Fetch Next Question' to retry.", [], "", ""

    # Isolate question text body
    q_text = f"### Question:\n{data['question']}"

    # CHANGED: Create the dynamic reference row cleanly as an isolated string
    file_name = data.get('source_file', 'Unknown PDF')
    page_num = data.get('source_page', 'Unknown Page')
    meta_text = f"📖 **Source:** `{file_name}` (Page `{page_num}`) | 🧠 **Model In Use:** `{quiz_engine.active_model_used}`"

    choices_list = [
        f"A: {data['options'].get('A', '')}",
        f"B: {data['options'].get('B', '')}",
        f"C: {data['options'].get('C', '')}",
        f"D: {data['options'].get('D', '')}"
    ]
    # Return 4 values to map directly to the expanded UI targets
    return q_text, gr.Radio(choices=choices_list, value=None), "", meta_text

def evaluate_selection(choice):
    if not quiz_engine.current_quiz:
        return "⚠️ Please load a question first."
    if not choice:
        return "⚠️ Please click an option bubble before submitting your selection."

    user_letter = choice.split(":")[0].strip().upper()
    correct_letter = quiz_engine.current_quiz['correct_answer'].strip().upper()

    if user_letter == correct_letter:
        score_text = "### ✅ CORRECT!\n\n"
    else:
        file_name = quiz_engine.current_quiz.get('source_file', 'Unknown PDF')
        page_num = quiz_engine.current_quiz.get('source_page', 'Unknown Page')
        score_text = f"### ❌ INCORRECT.\nThe correct answer choice was **({correct_letter})**.\n\n📍 *Double check this on page {page_num} of {file_name}*\n\n"
        quiz_engine.log_failure()

    return f"{score_text}📝 **Rationale Solution Breakdown**:\n{quiz_engine.current_quiz['explanation']}"

def view_summary_panel():
    return quiz_engine.get_summary()

# =====================================================================
# APPLICATION INTERFACE GENERATION CANVAS
# =====================================================================
with gr.Blocks(theme=gr.themes.Soft()) as mobile_app:
    gr.Markdown("# 🧠 Google Cloud GenAI Leader App Hub")

    with gr.Group():
        question_display = gr.Markdown("### Tap '➡️ Fetch Next Question' at the bottom to begin your practice run!")

    choice_radio = gr.Radio(choices=[], label="👇 Select your answer option:")
    submit_btn = gr.Button("🔒 Submit Selection", variant="primary")
    feedback_display = gr.Markdown("")

    # Pinned navigation controls layout row panel matrix
    with gr.Row():
        next_btn = gr.Button("➡️ Fetch Next Question", variant="secondary")
        summary_btn = gr.Button("📊 Weak Summary Dashboard", variant="stop")

    # CHANGED: Placed the tracking block explicitly BELOW the navigation row canvas
    metadata_tracker = gr.Markdown("")

    # Bind backend method updates sequentially to handle event changes smoothly
    next_btn.click(
        fn=load_new_question,
        inputs=[],
        # CHANGED: Added metadata_tracker as the fourth output channel parameter target
        outputs=[question_display, choice_radio, feedback_display, metadata_tracker],
        queue=True
    )

    submit_btn.click(fn=evaluate_selection, inputs=[choice_radio], outputs=[feedback_display])
    summary_btn.click(fn=view_summary_panel, inputs=[], outputs=[feedback_display])

# Run app interface
mobile_app.launch(inline=False, share=True, debug=False)

/tmp/ipykernel_4435/3583796088.py:178: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as mobile_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://41f2a045cbaa28bb94.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
